# Day 8: Snowflake Storage Integration & Dynamic Tables

This notebook uses DuckDB to simulate incremental query behavior and SQL transformation logic that would run via Snowflake Dynamic Tables.

In [ ]:
import duckdb
import pandas as pd

# Initialize in-memory DuckDB
con = duckdb.connect(database=':memory:')

In [ ]:
# Create Mock Silver Tables
con.execute("""
CREATE TABLE omop_visit_occurrence (
    person_id INTEGER,
    visit_occurrence_id INTEGER,
    visit_start_date DATE,
    visit_end_date DATE
)
""")

con.execute("""
INSERT INTO omop_visit_occurrence VALUES
    (1, 101, '2023-01-01', '2023-01-05'),
    (1, 102, '2023-01-15', '2023-01-16'), -- Readmission within 30 days
    (2, 201, '2023-02-01', '2023-02-10'),
    (2, 202, '2023-04-01', '2023-04-05')  -- Not a readmission within 30 days
""")

print("Silver omop_visit_occurrence:")
print(con.execute("SELECT * FROM omop_visit_occurrence").fetchdf())

In [ ]:
# Simulate Snowflake Dynamic Table Logic for 30-Day Readmissions
dynamic_table_sql = """
CREATE OR REPLACE VIEW readmission_metrics AS
WITH visits AS (
    SELECT 
        person_id,
        visit_occurrence_id,
        visit_start_date,
        visit_end_date,
        LAG(visit_end_date) OVER (PARTITION BY person_id ORDER BY visit_start_date) as prev_discharge_date
    FROM omop_visit_occurrence
)
SELECT
    person_id,
    COUNT(visit_occurrence_id) as total_visits,
    SUM(CASE WHEN prev_discharge_date IS NOT NULL AND (visit_start_date - prev_discharge_date) <= 30 THEN 1 ELSE 0 END) as readmissions_within_30_days
FROM visits
GROUP BY person_id
"""
con.execute(dynamic_table_sql)

print("\nSimulated Gold readmission_metrics:")
print(con.execute("SELECT * FROM readmission_metrics").fetchdf())